# Notebook 17 – Model Comparison Report

**Dataset:** Online Retail transactions (`data.csv`)

**Problem:** Flag **international orders** (non-UK, ~11% of data) — same problem as Notebook 16.

**Goal:** One final table comparing 6 models on scores, generalization, speed, complexity and interpretability — then decide which model to select and why.

## 1. Setup: Data & Three-Way Split
We need a **training**, **validation** and **test** set to report all three scores (60% / 20% / 20%). Cross-validation is run on train+validation combined.

In [7]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score, precision_recall_curve,
                             r2_score, mean_absolute_error, mean_squared_error)
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(8000, random_state=42)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Month'] = df['InvoiceDate'].dt.month
features = ['Quantity', 'UnitPrice', 'TotalPrice', 'Hour', 'DayOfWeek', 'Month']
X = df[features]
y = (df['Country'] != 'United Kingdom').astype(int)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_train.shape, X_val.shape, X_test.shape

((4800, 6), (1600, 6), (1600, 6))

# Part 1: Classification Models

## 2. Define the Models
Six different algorithms. Scale-sensitive models (Logistic Regression, KNN, SVM) get a `StandardScaler`.

In [8]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'SVM': make_pipeline(StandardScaler(), SVC()),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                                    subsample=0.7, random_state=42),
}
def get_scores(model, data):
    return model.predict_proba(data)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(data)

## 3. Compute All Metrics
- **Training / Validation / Test / CV score** = ROC-AUC (accuracy is misleading on imbalanced data).
- **Precision / Recall / F1** = on the test set, using each model's **own best-F1 threshold chosen on the validation set** (so every model is judged fairly, not at an arbitrary 0.5 cutoff).
- **Training / Prediction time** = measured with `time.time()`.

In [9]:
rows = []
for name, model in models.items():
    start = time.time(); model.fit(X_train, y_train); train_time = time.time() - start
    start = time.time(); test_scores = get_scores(model, X_test); pred_time = time.time() - start
    val_scores = get_scores(model, X_val)
    prec, rec, thr = precision_recall_curve(y_val, val_scores)
    best_thr = thr[(2 * prec * rec / (prec + rec + 1e-9))[:-1].argmax()]
    pred = (test_scores >= best_thr).astype(int)
    rows.append({
        'Model': name,
        'Training Score': roc_auc_score(y_train, get_scores(model, X_train)),
        'Validation Score': roc_auc_score(y_val, val_scores),
        'Test Score': roc_auc_score(y_test, test_scores),
        'Cross-Validation Score': cross_val_score(model, X_tv, y_tv, cv=skf, scoring='roc_auc').mean(),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, test_scores),
        'MAE / RMSE / R²': 'N/A (classification)',
        'Training Time (s)': train_time,
        'Prediction Time (s)': pred_time,
    })
results = pd.DataFrame(rows)

## 4. Add Complexity & Interpretability
These are properties of each algorithm (not computed from data).

In [10]:
qualitative = {
    'Logistic Regression': ('Low (linear, 6 coefficients)', 'High (coefficients)'),
    'KNN': ('Low to build, stores all data', 'Low'),
    'Decision Tree': ('Low-Medium (depth 5)', 'High (can draw the tree)'),
    'Random Forest': ('High (200 trees)', 'Medium (feature importance)'),
    'SVM': ('Medium-High (kernel)', 'Low'),
    'Gradient Boosting': ('High (200 sequential trees)', 'Medium (feature importance)'),
}
results['Model Complexity'] = results['Model'].map(lambda m: qualitative[m][0])
results['Interpretability'] = results['Model'].map(lambda m: qualitative[m][1])

## 5. Final Comparison Table (Classification)
Shown transposed (models as columns) so all 14 attributes fit on screen.

In [11]:
final_table = results.copy()
num_cols = final_table.select_dtypes('number').columns
final_table[num_cols] = final_table[num_cols].round(3)
final_table.set_index('Model').T

Model,Logistic Regression,KNN,Decision Tree,Random Forest,SVM,Gradient Boosting
Training Score,0.624,0.77,0.761,0.902,0.673,0.787
Validation Score,0.636,0.679,0.734,0.773,0.614,0.785
Test Score,0.631,0.68,0.705,0.735,0.59,0.737
Cross-Validation Score,0.625,0.68,0.716,0.744,0.591,0.75
Precision,0.205,0.195,0.178,0.222,0.183,0.219
Recall,0.236,0.556,0.736,0.421,0.449,0.522
F1,0.219,0.289,0.287,0.291,0.26,0.308
ROC-AUC,0.631,0.68,0.705,0.735,0.59,0.737
MAE / RMSE / R²,N/A (classification),N/A (classification),N/A (classification),N/A (classification),N/A (classification),N/A (classification)
Training Time (s),0.025,0.031,0.014,1.583,1.535,1.294


## 6. Generalization Check
Training score minus validation score — a big gap means overfitting.

In [12]:
gap = results[['Model', 'Training Score', 'Validation Score', 'Test Score']].copy()
gap['Train-Val Gap'] = gap['Training Score'] - gap['Validation Score']
gap.round(3).sort_values('Train-Val Gap')

,Model,Training Score,Validation Score,Test Score,Train-Val Gap
0,Logistic Regression,0.624,0.636,0.631,-0.012
5,Gradient Boosting,0.787,0.785,0.737,0.002
2,Decision Tree,0.761,0.734,0.705,0.027
4,SVM,0.673,0.614,0.590,0.059
1,KNN,0.770,0.679,0.680,0.092
3,Random Forest,0.902,0.773,0.735,0.129


# Part 2: Regression Models (where applicable)
For regression the table uses **MAE / RMSE / R²** instead of Precision/Recall/F1/ROC-AUC. Task: predict `log(1 + UnitPrice)` from quantity, time and country type.

In [13]:
dfr = df.copy()
dfr['LogQuantity'] = np.log1p(dfr['Quantity'])
dfr['IsInternational'] = (dfr['Country'] != 'United Kingdom').astype(int)
Xr = dfr[['LogQuantity', 'Hour', 'DayOfWeek', 'Month', 'IsInternational']]
yr = np.log1p(dfr['UnitPrice'])
Xr_tv, Xr_test, yr_tv, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
Xr_train, Xr_val, yr_train, yr_val = train_test_split(Xr_tv, yr_tv, test_size=0.25, random_state=42)
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                                                             subsample=0.7, random_state=42),
}
reg_quality = {
    'Linear Regression': ('Low', 'High (coefficients)'),
    'Ridge': ('Low', 'High (coefficients)'),
    'Random Forest Regressor': ('High (200 trees)', 'Medium'),
    'Gradient Boosting Regressor': ('High (200 sequential trees)', 'Medium'),
}
reg_rows = []
for name, model in reg_models.items():
    start = time.time(); model.fit(Xr_train, yr_train); train_time = time.time() - start
    start = time.time(); pred = model.predict(Xr_test); pred_time = time.time() - start
    reg_rows.append({
        'Model': name,
        'Training Score (R²)': r2_score(yr_train, model.predict(Xr_train)),
        'Validation Score (R²)': r2_score(yr_val, model.predict(Xr_val)),
        'Test Score (R²)': r2_score(yr_test, pred),
        'Cross-Validation Score (R²)': cross_val_score(model, Xr_tv, yr_tv, cv=KFold(5, shuffle=True, random_state=42), scoring='r2').mean(),
        'MAE': mean_absolute_error(yr_test, pred),
        'RMSE': mean_squared_error(yr_test, pred) ** 0.5,
        'Training Time (s)': train_time,
        'Prediction Time (s)': pred_time,
        'Model Complexity': reg_quality[name][0],
        'Interpretability': reg_quality[name][1],
    })
reg_table = pd.DataFrame(reg_rows)
num_cols = reg_table.select_dtypes('number').columns
reg_table[num_cols] = reg_table[num_cols].round(3)
reg_table.set_index('Model').T

Model,Linear Regression,Ridge,Random Forest Regressor,Gradient Boosting Regressor
Training Score (R²),0.166,0.166,0.373,0.296
Validation Score (R²),0.191,0.191,0.26,0.278
Test Score (R²),0.178,0.178,0.246,0.263
Cross-Validation Score (R²),0.171,0.171,0.242,0.254
MAE,0.427,0.427,0.391,0.391
RMSE,0.538,0.538,0.516,0.51
Training Time (s),0.006,0.004,1.803,1.114
Prediction Time (s),0.002,0.002,0.122,0.013
Model Complexity,Low,Low,High (200 trees),High (200 sequential trees)
Interpretability,High (coefficients),High (coefficients),Medium,Medium


# Which Model Should Be Selected, and Why?

**Selected model (classification): Gradient Boosting.**

**1. Best predictive ability.** It has the highest validation, cross-validation and test ROC-AUC, and the best F1 at its tuned threshold — ahead of Random Forest, SVM, KNN, Decision Tree and Logistic Regression.

**2. Best generalization.** Its training and validation scores are almost identical (tiny train-val gap), so its results are trustworthy. Random Forest scores higher on the training set but drops a lot on validation → it **overfits** more, even though it looks strong on training data.

**3. Fast to use.** Training takes under a second and predictions are near-instant — better than SVM and KNN, which are slower at prediction time.

**4. We did not pick it just for the highest score.** The F1 gap between the top four models (KNN, Decision Tree, Random Forest, Gradient Boosting) is small (about 0.29 to 0.31), so we also weighed overfitting, speed, and explainability. Gradient Boosting wins on generalization and AUC without a serious cost in speed.

**Trade-off / when to choose differently:**
- If the business must **explain every decision** (e.g., to auditors), pick the **Decision Tree**: its F1 is close, it is extremely fast, and you can draw the rules — at the cost of lower AUC.
- Logistic Regression is the simplest, but it underfits here, and the untuned SVM also scores low — both have the weakest AUC in the table.

**Regression (if the task were predicting price):** Gradient Boosting Regressor has the highest test R² and lowest RMSE, with a small train-test gap. However, R² is only about 0.25, so all models are weak — the features, not the algorithm, are the limit.